Importaremos o ecossistema e definimos a variável de controle do banco de dados gerado na Task 2.

In [6]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import awswrangler as wr
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuração do estilo visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Definição do banco de dados do blue
GLUE_DATABASE = "classicmodels_analytics"

# Define a região padrão para qualquer biblioteca da AWS que rodar neste Notebook
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

print(f"Sessão iniciada. Conectado ao banco analítico: {GLUE_DATABASE}")

Sessão iniciada. Conectado ao banco analítico: classicmodels_analytics


Consulta ao Athena para inspecionar o catálogo de produtos e garantir que o contrato de colunas da Task 2 está funcionando.

In [8]:
print("Executando consulta exploratória em dim_products...")

query_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

# Carrega o resultado diretamente do Athena para um DataFrame do Pandas
df_products_sample = wr.athena.read_sql_query(sql=query_products, database=GLUE_DATABASE)

# Exibe as primeiras linhas do catálogo de produtos
display(df_products_sample.head(10))

Executando consulta exploratória em dim_products...


,product_id,product_name,product_line,product_vendor
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,Min Lin Diecast
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,Classic Metal Creations
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,Highway 66 Mini Classics
3,S12_1108,2001 Ferrari Enzo,Classic Cars,Second Gear Diecast
4,S12_1666,1958 Setra Bus,Trucks and Buses,Welly Diecast Productions
5,S12_2823,2002 Suzuki XREO,Motorcycles,Unimax Art Galleries
6,S12_3148,1969 Corvair Monza,Classic Cars,Welly Diecast Productions
7,S12_3380,1968 Dodge Charger,Classic Cars,Welly Diecast Productions
8,S12_3891,1969 Ford Falcon,Classic Cars,Second Gear Diecast
9,S12_3990,1970 Plymouth Hemi Cuda,Classic Cars,Studio M Art Models


Executaremos a agregação volumétrica cruzando a tabela fato com a dimensão geográfica por meio da chave (country_key).

In [9]:
print("Calculando o ranking de vendas totais por país...")

query_countries = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_by_country = wr.athena.read_sql_query(sql=query_countries, database=GLUE_DATABASE)

# Exibe o resultado estruturado
display(df_sales_by_country)

Calculando o ranking de vendas totais por país...


c:\Users\T-Gamer\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,country,total_sales
0,USA,3273280.05
1,Spain,2198778.18
2,France,1007374.02
3,Singapore,791993.34
4,Australia,562582.59
5,New Zealand,476847.01
6,UK,436947.44
7,Germany,392941.98
8,Italy,360616.81
9,Finland,295149.35


Para otimizar a performance do dashboard interativo e evitar múltiplas consultas, extraímos a base e geramos um dataframe.

In [10]:
print("Extraindo base analítica detalhada para o Dashboard (Aguarde)...")

query_detailed = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

# Armazena a base bruta na memória local do Pandas
df_base_analytics = wr.athena.read_sql_query(sql=query_detailed, database=GLUE_DATABASE)

# REGRA DE NEGÓCIO: Converter a coluna de data para o tipo datetime do pandas
df_base_analytics['full_date'] = pd.to_datetime(df_base_analytics['full_date'])

print(f"Base analítica carregada com sucesso! Total de registros: {len(df_base_analytics)}")

Extraindo base analítica detalhada para o Dashboard (Aguarde)...


c:\Users\T-Gamer\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Base analítica carregada com sucesso! Total de registros: 2996


Constrói o painel utilizando o ipywidgets. Com filtros dinâmicos, tratando cenário global e ordenando pelo Top N escolhido pelo usuário.

In [ ]:
# Configuração dos widgets de controle para o dashboard

# Seletores de datas baseados nos limites reais dos dados extraídos
min_date = df_base_analytics['full_date'].min().date()
max_date = df_base_analytics['full_date'].max().date()

start_date_w = widgets.DatePicker(description='Data Início:', value=min_date)
end_date_w = widgets.DatePicker(description='Data Fim:', value=max_date)

# Seletores de Categorias com a opção agregadora 'Todos'
list_countries = ['Todos'] + sorted(df_base_analytics['country'].unique().tolist())
country_w = widgets.Dropdown(options=list_countries, value='Todos', description='País:')

list_lines = ['Todos'] + sorted(df_base_analytics['product_line'].unique().tolist())
product_line_w = widgets.Dropdown(options=list_lines, value='Todos', description='Linha Prod:')

# Slider para o controle dinâmico do Top N
top_n_w = widgets.IntSlider(value=5, min=1, max=10, step=1, description='Top N:')

# Container de saída para renderização do gráfico
output_plot = widgets.Output()

# Filtragem e renderização
def atualizar_dashboard(*args):
    with output_plot:
        clear_output(wait=True)
        
        # Copia a base para aplicar filtros sucessivos
        df_filtered = df_base_analytics.copy()
        
        # Filtro por Intervalo de Datas
        if start_date_w.value and end_date_w.value:
            t_start = pd.to_datetime(start_date_w.value)
            t_end = pd.to_datetime(end_date_w.value)
            df_filtered = df_filtered[(df_filtered['full_date'] >= t_start) & (df_filtered['full_date'] <= t_end)]
        
        # Filtro por País (Tratando a opção 'Todos')
        if country_w.value != 'Todos':
            df_filtered = df_filtered[df_filtered['country'] == country_w.value]
            
        # Filtro por Linha de Produto (Tratando a opção 'Todos')
        if product_line_w.value != 'Todos':
            df_filtered = df_filtered[df_filtered['product_line'] == product_line_w.value]
            
        # Agregação e Ranquamento (Explorar -> Agregar -> Ranquear)
        if df_filtered.empty:
            print("Nenhum registro encontrado para as combinações de filtros selecionadas.")
            return
            
        df_grouped = df_filtered.groupby('product_name')['total_sales'].sum().reset_index()
        df_top = df_grouped.sort_values(by='total_sales', ascending=False).head(top_n_w.value)
        
        # Construção do Gráfico de Barras Horizontal (Seaborn)
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.barplot(
            data=df_top,
            x='total_sales',
            y='product_name',
            hue='product_name',
            palette='viridis',
            ax=ax,
            legend=False
        )
        
        # Ajustes de layout e formatação de moeda
        ax.set_title(f"Top {top_n_w.value} Produtos em Faturamento (Sales Amount)", fontsize=14, weight='bold')
        ax.set_xlabel("Vendas Totais ($)", fontsize=11)
        ax.set_ylabel("Nome do Produto", fontsize=11)
        
        # Adiciona rótulos de valores no final das barras para facilitar a leitura rápida
        for container in ax.containers:
            ax.bar_label(container, fmt='$%1.2f', padding=5, fontsize=9)
            
        plt.tight_layout()
        plt.show()

# Vincular a função de atualização às mudanças de estado dos widgets
start_date_w.observe(atualizar_dashboard, 'value')
end_date_w.observe(atualizar_dashboard, 'value')
country_w.observe(atualizar_dashboard, 'value')
product_line_w.observe(atualizar_dashboard, 'value')
top_n_w.observe(atualizar_dashboard, 'value')

# Painel
# Organiza os controles visualmente em caixas estruturadas
controles_data = widgets.HBox([start_date_w, end_date_w])
controles_filtros = widgets.HBox([country_w, product_line_w, top_n_w])
dashboard_ui = widgets.VBox([controles_data, controles_filtros, output_plot])

display(dashboard_ui)

# Executa a renderização inicial do gráfico assim que o notebook carrega
atualizar_dashboard()